In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
df=pd.read_csv('/kaggle/input/competitions/titanic/train.csv')

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Separating Features and Target

In [4]:
y=df['Survived']
x=df.copy()#For now keeping all colms in x for analysis

In [5]:
x.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [6]:
x.drop(columns=['PassengerId','Cabin','Ticket'],inplace=True)#Not using for prediction

In [7]:
x.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S


# Feature Splitting

In [8]:
x['Name']=x['Name'].str.split(',',expand=True)[1].str.split(expand=True)[0]

In [9]:
name_survival_percent=pd.DataFrame(round(x.groupby('Name')['Survived'].mean()*100,2))

In [10]:
name_survival_percent

,Survived
Name,
Capt.,0.00
Col.,50.00
Don.,0.00
Dr.,42.86
Jonkheer.,0.00
Lady.,100.00
Major.,50.00
Master.,57.50
Miss.,69.78


**Custom Encoding Name Colm**

In [11]:
name_survival_percent.loc['Mr.'].values

array([15.67])

In [12]:
#0-20 -> A
#20-80 -> B
#80-100 -> C

def custom_encoder(x):
    val = name_survival_percent.loc[x].values
    if(val >=0 and val<=20):
        return "A"
    elif(val>20 and val<= 80):
        return "B"
    else:
        return "C"
x['Name']=x['Name'].apply(custom_encoder)  

In [13]:
x.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,A,male,22.0,1,0,7.2500,S
1,1,1,B,female,38.0,1,0,71.2833,C
2,1,3,B,female,26.0,0,0,7.9250,S
3,1,1,B,female,35.0,1,0,53.1000,S
4,0,3,A,male,35.0,0,0,8.0500,S


# Feature Construction

In [14]:
x['Family']=x['SibSp']+x['Parch']+1

In [15]:
x.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,Family
0,0,3,A,male,22.0,1,0,7.2500,S,2
1,1,1,B,female,38.0,1,0,71.2833,C,2
2,1,3,B,female,26.0,0,0,7.9250,S,1
3,1,1,B,female,35.0,1,0,53.1000,S,2
4,0,3,A,male,35.0,0,0,8.0500,S,1


In [16]:
family_survival_percent=pd.DataFrame(round(x.groupby('Family')['Survived'].mean()*100,2))


In [17]:
family_survival_percent

,Survived
Family,
1,30.35
2,55.28
3,57.84
4,72.41
5,20.00
6,13.64
7,33.33
8,0.00
11,0.00


**Custom Encoding Family Colm**

In [18]:
#0 -> 0
#10 - 20 -> 1
#30-40 -> 2
#>=50 -> 3

def custom_family_encoder(x):
    val = family_survival_percent.loc[x].values
    if(val == 0):
        return 0
    elif(val>=10 and val<= 20):
        return 1
    elif(val>=30 and val<=40):
        return 2
    else:
        return 3
x['Family']=x['Family'].apply(custom_family_encoder)  

In [19]:
x.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,Family
0,0,3,A,male,22.0,1,0,7.2500,S,3
1,1,1,B,female,38.0,1,0,71.2833,C,3
2,1,3,B,female,26.0,0,0,7.9250,S,2
3,1,1,B,female,35.0,1,0,53.1000,S,3
4,0,3,A,male,35.0,0,0,8.0500,S,2


In [20]:
x.drop(columns=['Survived','SibSp','Parch'],inplace=True)

In [21]:
x.head()

,Pclass,Name,Sex,Age,Fare,Embarked,Family
0,3,A,male,22.0,7.2500,S,3
1,1,B,female,38.0,71.2833,C,3
2,3,B,female,26.0,7.9250,S,2
3,1,B,female,35.0,53.1000,S,3
4,3,A,male,35.0,8.0500,S,2


# Designing the Architecture

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer,KNNImputer,IterativeImputer

In [23]:
embarked_pipeline=Pipeline(steps=[
    ('emb_imp',SimpleImputer(strategy='most_frequent')),
    ('emb_enc',OneHotEncoder(sparse_output=False,drop='first'))
])

In [24]:
name_sex_pipeline=Pipeline(steps=[
    ('name_sex_enc',OneHotEncoder(sparse_output=False,drop='first'))
])

In [25]:
categorical_processor=ColumnTransformer(transformers=[
    ('embarked_pipeline',embarked_pipeline,['Embarked']),
    ('name_sex_pipeline',name_sex_pipeline,['Name','Sex'])
],remainder='passthrough')

In [26]:
from sklearn.preprocessing import PowerTransformer

In [27]:
preprocessor=Pipeline(steps=[
    ('categorical_processor',categorical_processor),
    ('iter_imp',IterativeImputer()),
    ('ptnf',PowerTransformer())
])

In [28]:
from sklearn.neighbors import KNeighborsClassifier

In [29]:
classifier=KNeighborsClassifier()

In [30]:
model=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('classifier',classifier)
])

# Model Evaluation

In [31]:
from sklearn.linear_model import BayesianRidge
from sklearn.tree import ExtraTreeRegressor,DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

In [32]:
parameter_settings={
    'preprocessor__iter_imp__estimator':[BayesianRidge(),RandomForestRegressor(),ExtraTreeRegressor(),KNeighborsRegressor(),DecisionTreeRegressor()],
    'classifier__n_neighbors':[x for x in range(2,15)]
}

In [33]:
from sklearn.model_selection import GridSearchCV

In [34]:
grid_search=GridSearchCV(model,parameter_settings,scoring='accuracy',cv=10)

In [35]:
grid_search.fit(x,y)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warning

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('preprocessor',
                                        Pipeline(steps=[('categorical_processor',
                                                         ColumnTransformer(remainder='passthrough',
                                                                           transformers=[('embarked_pipeline',
                                                                                          Pipeline(steps=[('emb_imp',
                                                                                                           SimpleImputer(strategy='most_frequent')),
                                                                                                          ('emb_enc',
                                                                                                           OneHotEncoder(drop='first',
                                                                                                                         sparse_output=False))]),
                                                                                          ['Embarked']),
                                                                                         ('name_sex_pipeline',
                                                                                          Pipeli...
                                                                                           'Sex'])])),
                                                        ('iter_imp',
                                                         IterativeImputer()),
                                                        ('ptnf',
                                                         PowerTransformer())])),
                                       ('classifier', KNeighborsClassifier())]),
             param_grid={'classifier__n_neighbors': [2, 3, 4, 5, 6, 7, 8, 9, 10,
                                                     11, 12, 13, 14],
                         'preprocessor__iter_imp__estimator': [BayesianRidge(),
                                                               RandomForestRegressor(),
                                                               ExtraTreeRegressor(),
                                                               KNeighborsRegressor(),
                                                               DecisionTreeRegressor()]},
             scoring='accuracy')

In [36]:
grid_search.best_score_

np.float64(0.8372784019975033)

In [37]:
grid_search.best_params_

{'classifier__n_neighbors': 10,
 'preprocessor__iter_imp__estimator': RandomForestRegressor()}

# Testing


In [38]:
test_data = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

In [39]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [40]:
test_data['Name']=test_data['Name'].str.split(',',expand=True)[1].str.split(expand=True)[0]

In [41]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,Mr.,male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,Mrs.,female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,Mr.,male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,Mr.,male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,Mrs.,female,22.0,1,1,3101298,12.2875,NaN,S


In [42]:
test_data[test_data['Name']=='Dona.']

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
414,1306,1,Dona.,female,39.0,0,0,PC 17758,108.9,C105,C


In [43]:
test_data['Name'].value_counts()

Name
Mr.        240
Miss.       78
Mrs.        72
Master.     21
Col.         2
Rev.         2
Ms.          1
Dr.          1
Dona.        1
Name: count, dtype: int64

In [44]:
def custom_encoder(x):
    if(x=='Dona.'):
        return "A"
    val = name_survival_percent.loc[x].values
    if(val >=0 and val<=20):
        return "A"
    elif(val>20 and val<= 80):
        return "B"
    else:
        return "C"
test_data['Name']=test_data['Name'].apply(custom_encoder)

In [45]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,A,male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,B,female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,A,male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,A,male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,B,female,22.0,1,1,3101298,12.2875,NaN,S


In [46]:
test_data['Family']=test_data['SibSp']+test_data['Parch']+1

In [47]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family
0,892,3,A,male,34.5,0,0,330911,7.8292,NaN,Q,1
1,893,3,B,female,47.0,1,0,363272,7.0000,NaN,S,2
2,894,2,A,male,62.0,0,0,240276,9.6875,NaN,Q,1
3,895,3,A,male,27.0,0,0,315154,8.6625,NaN,S,1
4,896,3,B,female,22.0,1,1,3101298,12.2875,NaN,S,3


In [48]:
def custom_family_encoder(x):
    val = family_survival_percent.loc[x].values
    if(val == 0):
        return 0
    elif(val>=10 and val<= 20):
        return 1
    elif(val>=30 and val<=40):
        return 2
    else:
        return 3
test_data['Family']=test_data['Family'].apply(custom_family_encoder)  

In [49]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family
0,892,3,A,male,34.5,0,0,330911,7.8292,NaN,Q,2
1,893,3,B,female,47.0,1,0,363272,7.0000,NaN,S,3
2,894,2,A,male,62.0,0,0,240276,9.6875,NaN,Q,2
3,895,3,A,male,27.0,0,0,315154,8.6625,NaN,S,2
4,896,3,B,female,22.0,1,1,3101298,12.2875,NaN,S,3


In [50]:
test_data_final=test_data.drop(columns=['PassengerId','SibSp','Parch','Ticket','Cabin'],inplace=True)

In [51]:
test_data.head()

,Pclass,Name,Sex,Age,Fare,Embarked,Family
0,3,A,male,34.5,7.8292,Q,2
1,3,B,female,47.0,7.0000,S,3
2,2,A,male,62.0,9.6875,Q,2
3,3,A,male,27.0,8.6625,S,2
4,3,B,female,22.0,12.2875,S,3


In [52]:
y_pred=grid_search.predict(test_data)

In [53]:
test_data2 = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')


In [54]:
test_data2

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [55]:
output=pd.DataFrame()

In [56]:
output['PassengerId']=test_data2['PassengerId']
output['Survived']=y_pred

In [57]:
output

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [58]:
output.to_csv('output.csv',index=False)